<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.9em; font-weight:bold; margin:0.6em 0 0.3em 0;">מחברת 3.4: בוחרים K בעזרת ולידציה צולבת</p>

<p dir="rtl" style="direction:rtl; text-align:right;">זהו המשך מודרך של חלק ד במפגש 3. ב-K-NN מספר השכנים K הוא <b>היפר-פרמטר</b>: מספר שאנחנו קובעים לפני האימון, והמודל לא לומד אותו. איך בוחרים אותו בלי להציץ בקבוצת הבדיקה? <b>ולידציה צולבת</b>: מחלקים את קבוצת האימון לחלקים, וכל חלק בתורו משמש לבדיקה. נבחר K בעזרתה, ורק בסוף נבדוק פעם אחת על קבוצת הבדיקה. המטרה: הציון הכרוני <code dir="ltr">fma_sum_chr</code> (רגרסיה). משך מוערך: כ-10 דקות. בכל תא פקודה אחת שמדפיסה תוצאה אחת.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>כך עובד כל תא בדיקה:</b> אם התשובה נכונה יודפס <code dir="ltr">OK</code>. אם לא, תופיע שגיאה, וזה בסדר: תקנו והריצו שוב. כל מקום שבו כתוב <code dir="ltr">___</code> מחכה להשלמה שלכם.</p>

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">הכנת הסביבה</p>

<p dir="rtl" style="direction:rtl; text-align:right;">התא הבא מתקין רק את החבילות הדרושות (<code dir="ltr">tidymodels</code> ו-<code dir="ltr">kknn</code>) ממאגר בינארי מהיר, ומדפיס כמה זמן ההתקנה נמשכה. הריצו אותו פעם אחת בתחילת המפגש.</p>

In [ ]:
# Install only the packages the course needs, from a binary repository (fast)
options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
options(HTTPUserAgent = sprintf("R/%s R (%s)", getRversion(),
        paste(getRversion(), R.version$platform, R.version$arch, R.version$os)))
# tidymodels + kknn: about 100 packages in all, 43 more than tidyverse alone. From the Posit binary
# repository there is no compiling, but the time on a fresh Colab session was NOT measured here
# (a rough guess is a few minutes). Run this cell first, before class starts.
needed  <- c("tidymodels", "kknn")   # add packages here when a notebook needs them
missing <- setdiff(needed, rownames(installed.packages()))   # skip installed ones
t <- system.time(if (length(missing) > 0) install.packages(missing))
cat("Installed:", if (length(missing)) paste(missing, collapse = ", ") else "nothing new",
    "| elapsed seconds:", round(t[["elapsed"]], 1), "\n")
suppressPackageStartupMessages(library(tidymodels))   # recipes, parsnip, rsample, yardstick, tune, ggplot2, dplyr ...
tidymodels_prefer()                                   # when two packages share a name, prefer the tidymodels one

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">הכנה מוכנה מראש (ממחברת 3.1)</p>

<p dir="rtl" style="direction:rtl; text-align:right;">טעינה, פיצול (אותו <code dir="ltr">set.seed(820)</code>) ומתכון, כמו בשתי המחברות הקודמות. הפעם לא נריץ <code dir="ltr">prep()</code> ו-<code dir="ltr">bake()</code> בעצמנו: המתכון ייכנס ל"זרימת עבודה" שתלמד אותו מחדש בתוך כל חלק אימון.</p>

In [ ]:
source("https://raw.githubusercontent.com/ShayOfir/DSAI4PT_Zfat_Students/main/R/load_course_data.R")

In [ ]:
stroke <- load_course_data("stroke_clean") |>        # the cleaned table: one row per patient
  as_tibble() |>                                      # make it a tibble
  mutate(sex = factor(sex, levels = c("F", "M")),     # categories are stored as factors
         lesion_type = factor(lesion_type, levels = c("ischemic", "hemorrhagic", "mixed")))

stroke_ml <- stroke |>
  select(id, fma_sum_chr,                                   # id and the chronic score (the target)
         fma_sum_sub, fma_a_sub, fma_bc_sub,                # features known at the subacute visit ...
         age, sex, lesion_type, lesion_vol, tao_sub) |>     # ... as in chapter 8 of the book
  mutate(outcome_mild = factor(if_else(fma_sum_chr >= 43, "mild", "not_mild"),   # the categorical target
                               levels = c("mild", "not_mild")))

In [ ]:
set.seed(820)                                                 # fixed seed: everyone gets the same split
stroke_split <- initial_split(stroke_ml, prop = 0.75, strata = outcome_mild)   # 75% training, 25% test
stroke_train <- training(stroke_split)                        # the training table
stroke_test  <- testing(stroke_split)                         # the test table (locked until the end)

In [ ]:
stroke_rec <- recipe(fma_sum_chr ~ fma_sum_sub + age + sex + lesion_vol,
                     data = stroke_train) |>      # the plan: which columns, and which is the target
  step_impute_mode(sex) |>                         # fill a missing category with the most common one
  step_log(lesion_vol) |>                          # log of the lesion volume (it is very skewed)
  step_normalize(all_numeric_predictors()) |>      # z-score for every numeric predictor
  step_dummy(all_nominal_predictors())             # categories -> 0/1 columns

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 1: מחלקים את האימון לחלקים</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><code dir="ltr">vfold_cv(data, v = 5)</code> מחלקת את הטבלה באקראי ל-5 חלקים (folds). הארגומנט <code dir="ltr">strata = outcome_mild</code> שומר שבכל חלק יהיה שיעור דומה של "ליקוי קל", כמו בפיצול הראשי. בכל פיצול ארבעה חלקים משמשים לאימון (<code dir="ltr">analysis</code>) והחלק החמישי לבדיקה (<code dir="ltr">assessment</code>). בספר חילקנו ל-10 חלקים, אבל כאן באימון רק 36 מטופלים, ולכן 5 חלקים נותנים 7-8 מטופלים בכל חלק בדיקה. חשוב: מחלקים את <b>האימון בלבד</b>, לא את כל הנתונים.</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> חלקו את <code dir="ltr">stroke_train</code> ל-5 חלקים.</p>

In [ ]:
set.seed(801)                                       # fixed seed: the same random folds for everyone
stroke_folds <- vfold_cv(___, v = ___, strata = outcome_mild)   # TODO: which table, and how many folds?
stroke_folds

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(stroke_folds) == 5, nrow(stroke_folds$splits[[1]]$data) == 36)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;">כל שורה בטבלה היא פיצול אחד (<code dir="ltr">id</code> = שם החלק). הפונקציות <code dir="ltr">analysis()</code> ו-<code dir="ltr">assessment()</code> שולפות ממנו את מטופלי האימון ואת מטופלי הבדיקה. דוגמה: כמה מטופלים יש באימון בפיצול הראשון? (<code dir="ltr">[[1]]</code> שולף את הרכיב הראשון מרשימה.)</p>

In [ ]:
first_split <- stroke_folds$splits[[1]]   # the first of the 5 splits
nrow(analysis(first_split))                  # patients used for training in this round

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף ב.</b> כמה מטופלים נבדקים בפיצול הראשון (<code dir="ltr">assessment()</code>)?</p>

In [ ]:
n_assess <- nrow(___(first_split))   # TODO: the function that returns the held-out patients
n_assess

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(n_assess == nrow(stroke_folds$splits[[1]]$data) - nrow(analysis(first_split)), n_assess %in% c(7, 8))
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 2: זרימת עבודה ושגיאה עבור K אחד</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><b>זרימת עבודה (workflow)</b> אורזת יחד את המתכון ואת המודל לאובייקט אחד. כך בכל סבב ההכנה (השלמה, תקנון, דמה) נלמדת מחלק האימון של אותו סבב בלבד, ולא מהחלק שנבדק: זה בדיוק הכלל ממחברת 3.1, והוא מתבצע אוטומטית. <code dir="ltr">workflow() |&gt; add_recipe() |&gt; add_model()</code>. דוגמה עם K = 5:</p>

In [ ]:
knn_spec5 <- nearest_neighbor(neighbors = 5, weight_func = "rectangular") |>   # K = 5
  set_mode("regression")
knn_wf5 <- workflow() |>          # an empty workflow
  add_recipe(stroke_rec) |>       # add the data preparation
  add_model(knn_spec5)            # add the model
knn_wf5

<p dir="rtl" style="direction:rtl; text-align:right;"><code dir="ltr">fit_resamples()</code> מאמנת ומעריכה את זרימת העבודה בכל הפיצולים, ו-<code dir="ltr">collect_metrics()</code> מסכמת: הממוצע של ה-<span dir="ltr">RMSE</span> על פני הפיצולים (<code dir="ltr">mean</code>) וטעות התקן (<code dir="ltr">std_err</code>). דוגמה:</p>

In [ ]:
cv5 <- fit_resamples(knn_wf5, resamples = stroke_folds, metrics = metric_set(rmse))   # fit and evaluate in all 5 folds
collect_metrics(cv5)                                                                      # the mean RMSE over the 5 folds

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> עשו אותו דבר עבור K = 15 והשוו: האם ה-RMSE הממוצע עלה או ירד?</p>

In [ ]:
knn_spec15 <- nearest_neighbor(neighbors = ___, weight_func = "rectangular") |>   # TODO: K = 15
  set_mode("regression")
knn_wf15 <- workflow() |> add_recipe(stroke_rec) |> add_model(knn_spec15)   # the same recipe, a different K
cv15 <- fit_resamples(knn_wf15, resamples = stroke_folds, metrics = metric_set(rmse))
collect_metrics(cv15)

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(collect_metrics(cv15)) == 1, nrow(collect_metrics(cv5)) == 1,
          rlang::eval_tidy(knn_spec15$args$neighbors) == 15)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 3: רשת של ערכי K</p>

<p dir="rtl" style="direction:rtl; text-align:right;">במקום לחזור ידנית על K אחרי K (בספר עשינו זאת בלולאת <code dir="ltr">for</code>), מסמנים בספק של המודל <code dir="ltr">neighbors = tune()</code> ("עדיין לא ידוע"), ונותנים ל-<code dir="ltr">tune_grid()</code> לעבור על רשימת ערכים. הרשת היא טבלה עם עמודה אחת בשם ההיפר-פרמטר.</p>

In [ ]:
knn_spec <- nearest_neighbor(neighbors = tune(), weight_func = "rectangular") |>   # K is left open
  set_mode("regression")
knn_wf <- workflow() |> add_recipe(stroke_rec) |> add_model(knn_spec)                  # recipe + model, K still open
k_grid <- tibble(neighbors = c(1, 2, 3, 5, 7, 10, 15, 20))                               # the values of K to try
k_grid

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> הריצו את החיפוש: <code dir="ltr">tune_grid()</code> מקבלת את זרימת העבודה, את החלקים ואת הרשת.</p>

In [ ]:
set.seed(802)                                                       # seed for the tuning run
knn_tuned <- tune_grid(knn_wf, resamples = ___, grid = ___,   # TODO: the folds and the grid
                       metrics = metric_set(rmse))   # 8 values of K x 5 folds = 40 fits

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(collect_metrics(knn_tuned)) == 8)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;">הטבלה: שורה לכל K, עם ה-<span dir="ltr">RMSE</span> הממוצע על פני 5 החלקים.</p>

In [ ]:
collect_metrics(knn_tuned) |> select(neighbors, mean, std_err)   # one row per K

<p dir="rtl" style="direction:rtl; text-align:right;">וגרף: <code dir="ltr">autoplot()</code> מציירת את ה-RMSE כפונקציה של K. הצורה המוכרת: שגיאה גבוהה כש-K קטן מדי (השונות), ויורדת, ואז עולה שוב כש-K גדול מדי (ההטיה).</p>

In [ ]:
autoplot(knn_tuned)   # validation RMSE for each K

<p dir="rtl" style="direction:rtl; text-align:right; font-size:1.5em; font-weight:bold; margin:0.6em 0 0.3em 0;">משימה 4: בוחרים K ובודקים פעם אחת</p>

<p dir="rtl" style="direction:rtl; text-align:right;"><code dir="ltr">select_best()</code> בוחרת את ה-K עם ה-RMSE הנמוך ביותר בוולידציה. <code dir="ltr">show_best()</code> מראה כמה מועמדים טובים.</p>

In [ ]:
show_best(knn_tuned, metric = "rmse", n = 3) |> select(neighbors, mean, std_err)   # the 3 best values of K

<p dir="rtl" style="direction:rtl; text-align:right;"><b>סעיף א.</b> שמרו את ה-K הטוב ביותר ב-<code dir="ltr">best_k</code>.</p>

In [ ]:
best_k <- select_best(knn_tuned, metric = "___")   # TODO: which metric did we tune on?
best_k

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(best_k$neighbors %in% k_grid$neighbors,
          best_k$neighbors == collect_metrics(knn_tuned)$neighbors[which.min(collect_metrics(knn_tuned)$mean)])
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;">עכשיו הצעד הסופי, <b>פעם אחת בלבד</b>: <code dir="ltr">finalize_workflow()</code> ממלאת את ה-K שנבחר בזרימת העבודה, ו-<code dir="ltr">last_fit()</code> מאמנת על כל קבוצת האימון ובודקת על קבוצת הבדיקה (היא מקבלת את הפיצול המקורי, <code dir="ltr">stroke_split</code>).</p>

In [ ]:
final_wf <- finalize_workflow(knn_wf, best_k)   # fill in the chosen K
final_fit <- last_fit(final_wf, ___,                    # TODO: the original split object
                      metrics = metric_set(rmse, mae, rsq))   # train on all training data, test once
final_fit |> collect_metrics()                                # the final test-set result

In [ ]:
# Visible check: stops with an error if the answer is wrong
stopifnot(nrow(collect_predictions(final_fit)) == 13, nrow(collect_metrics(final_fit)) == 3)
cat("OK\n")

<p dir="rtl" style="direction:rtl; text-align:right;"><b>תובנה.</b> איזה K נבחר, והאם תוצאת הבדיקה הסופית קרובה לשגיאת הוולידציה של אותו K? ומדוע אסור עכשיו, אחרי שראינו את תוצאת הבדיקה, לנסות K אחר "ליתר ביטחון"?</p>
<p dir="rtl" style="direction:rtl; text-align:right;">כתבו כאן משפט או שניים: ...</p>